In [22]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


from mtflearn import ZPs
from mtflearn.features import construct_rot_maps_matrix


def compute_kernels_and_linear_weights(n_max, size, n_folds=[2, 3, 4, 6]):
    zps = ZPs(n_max=n_max, size=size)
    kernels = zps.polynomials
    linear_weights = construct_rot_maps_matrix(n_folds, zps.m)
    return kernels, linear_weights

class FixedConvLayer(nn.Module):
    def __init__(self, kernels: torch.Tensor, linear_weight: torch.Tensor, stride=1):
        super(FixedConvLayer, self).__init__()
        assert kernels.shape[1] % 2 == 1 and kernels.shape[2] % 2 == 1, "Kernel size must be odd to preserve input shape."
        self.register_buffer('kernels', kernels.unsqueeze(1))  # Shape becomes (N, 1, H, W)
        self.stride = stride
        self.kernel_size = (kernels.shape[1], kernels.shape[2])  # (H, W)
        self.padding = ((self.kernel_size[0] - 1) // 2, (self.kernel_size[1] - 1) // 2)  # (pad_H, pad_W)
        self.norm_factor = 4./(self.kernel_size[0] * self.kernel_size[1]) / np.pi  # normalizingh factor

        # Linear layer with fixed weight
        self.register_buffer('linear_weight', linear_weight)  # Store as non-trainable

    def forward(self, x):
        if x.dim() == 2:  # Single image case (H, W)
            x = x.unsqueeze(0).unsqueeze(0)  # Shape becomes (1, 1, H, W)
        elif x.dim() == 3:  # Stack of images (num_imgs, H, W)
            x = x.unsqueeze(1)  # Shape becomes (num_imgs, 1, H, W)
        else:
            raise ValueError("Input must be of shape (H, W) or (num_imgs, H, W)")

        x1 = self.norm_factor * F.conv2d(x, self.kernels, stride=self.stride, padding=self.padding)
        x2 = x1 ** 2    # Shape becomes (num_imgs, num_kernels, H, W)

        # Reshape x2 to (num_imgs, num_kernels, H*W)
        x2_flat = x2.view(x2.shape[0], x2.shape[1], -1)  # (num_imgs, num_kernels, H*W)

        # Apply linear transformation: (another_num, num_kernels) @ (num_imgs, num_kernels, H*W) -> (num_imgs, another_num, H*W)
        x3 = torch.matmul(self.linear_weight, x2_flat)  # (num_imgs, another_num, H*W)

        # Reshape back to (num_imgs, another_num, H, W)
        x3 = x3.view(x2.shape[0], -1, x2.shape[2], x2.shape[3])

        return x3

## load data

In [23]:
import os
dp = r'D:\Dropbox\released projects\symmetry-learn\Fig5 symmetry maps\data'
filename = os.path.join(dp,  'monolayer_MoSe2_80K.npy')
img = np.load(filename)

In [24]:
n_max = 10
size = 75
kernels, weights = compute_kernels_and_linear_weights(n_max, size, n_folds=[2, 3, 4, 6])

kernels.shape, weights.shape

((66, 75, 75), (4, 66))

In [25]:
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [27]:
# numpy array to tensor
torch_kernels = torch.from_numpy(kernels).float().to(device)
torch_weights = torch.from_numpy(weights).float().to(device)

torch_img = torch.from_numpy(img).float().to(device)

fixed_conv = FixedConvLayer(torch_kernels, torch_weights).to(device)

output = fixed_conv(torch_img)
print(output.shape)  # Should be (N, H, W) -> (2, 5, 5) with padding ensuring input size is preserved

torch.Size([1, 4, 2048, 2048])
